In [1]:
from pathlib import Path
import csv
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F

### **Preprocessing**

In [19]:
folders = ['train', 'test']

with open(f'dataset.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['text', 'label'])

    for folder in folders:
            for idx, class_ in enumerate(['pos', 'neg']):
                files = list(Path().joinpath(f'aclImdb/{folder}/{class_}').iterdir())

                for file in files:
                    with open(file, 'r', encoding='utf-8') as ff:
                        line = ff.readlines()
                        line = line[0].replace('\n', '').replace('\t', '')

                        writer.writerow([line, idx])

### **Creating the DataFrame**

In [2]:
dataset = pd.read_csv('dataset.csv')
dataset = dataset.sample(frac=1, random_state=21).reset_index(drop=True)

In [3]:
train_dataframe, test_dataframe = train_test_split(dataset,
                                                   test_size=0.3,
                                                   stratify=dataset['label'],
                                                   random_state=21)

In [4]:
train_dataframe, val_dataframe = train_test_split(train_dataframe,
                                                  test_size=0.1,
                                                  stratify=train_dataframe['label'],
                                                  random_state=21)

### **Bag-of-Words Model**

In [22]:
cv = CountVectorizer(lowercase=True, 
                     max_features=10_000,
                     stop_words='english')

In [23]:
cv.fit(train_dataframe['text'])

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [24]:
x_train = cv.transform(train_dataframe['text'])
x_test = cv.transform(test_dataframe['text'])
x_val = cv.transform(val_dataframe['text'])

### **Creating the Dataset**

In [25]:
class TextDataset(Dataset):

    def __init__(self, x, y):
        super().__init__()

        self.features = torch.tensor(x, dtype=torch.float32)
        self.labels = torch.tensor(y, dtype=torch.float32)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return self.labels.shape[0]

In [26]:
train_dataset = TextDataset(x_train.todense(),
                            train_dataframe['label'].values)

In [27]:
test_dataset = TextDataset(x_test.todense(),
                           test_dataframe['label'].values)

In [28]:
val_dataset = TextDataset(x_val.todense(),
                          val_dataframe['label'].values)

### **Creating the DataLoader**

In [29]:
torch.manual_seed(21)

In [30]:
train_dataloader = DataLoader(train_dataset,
                              batch_size=32,
                              shuffle=True,
                              num_workers=0)

In [31]:
test_dataloader = DataLoader(test_dataset,
                             batch_size=32,
                             shuffle=False,
                             num_workers=0)

In [32]:
val_dataloader = DataLoader(val_dataset,
                            batch_size=32,
                            shuffle=False,
                            num_workers=0)

### **Logistic Classifier**

In [63]:
class LogisticClassifier(torch.nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()
        
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(in_features,5_000),
            torch.nn.ReLU(),

            torch.nn.Linear(5_000, 2_500),
            torch.nn.ReLU(),
            torch.nn.Dropout1d(0.2),

            torch.nn.Linear(2_500,1_000),
            torch.nn.ReLU(),

            torch.nn.Linear(1_000, out_features)
        )
    
    def forward(self, x):
        return torch.sigmoid(self.layers(x))

In [64]:
classifier = LogisticClassifier(10_000, 1).to('cuda')

### **Training**

In [65]:
optim = torch.optim.SGD(classifier.parameters(), lr=0.001, momentum=0.9 , weight_decay=1e-4)

In [66]:
epochs = 10

In [67]:
def accuracy(model, dataloader):
    model.eval()

    acc = 0.0
    total = 0
    for x, y in dataloader:
        x = x.to('cuda')

        with torch.inference_mode():
            a = model(x)
        
        a = torch.where(a > 0.5, 1, 0).to('cpu')
        labels = y.view(a.shape).to(a.dtype)

        result = a == labels
        acc += torch.sum(result).item()
        total += result.numel()
    
    return acc/total

In [68]:
best_loss = float('inf')
for epoch in range(epochs):
    classifier.train()

    total_loss = 0.0
    num_samples = 0
    for batch, (x, y) in enumerate(train_dataloader):
        x = x.to('cuda')
        y = y.to('cuda')

        a = classifier(x)

        loss = F.binary_cross_entropy(a, y.view(a.shape))

        batch_size = y.size(0)
        
        total_loss += loss.item() * batch_size
        num_samples += batch_size

        optim.zero_grad()
        loss.backward()
        optim.step()
    
    total_loss /= num_samples

    print(f'Epoch: {epoch + 1:03d}/{epochs:03d} - Loss: {total_loss: .6f}')

    if total_loss < best_loss:
        best_loss = total_loss

        torch.save(classifier.state_dict(), 'weights.pth')
    
    print(f'Train accuracy: {accuracy(classifier, train_dataloader)*100:.2f}% - Val accuracy: {accuracy(classifier, val_dataloader)*100: .2f}%')

Epoch: 001/010 - Loss:  0.691540
Train accuracy: 66.28% - Val accuracy:  66.03%
Epoch: 002/010 - Loss:  0.678661
Train accuracy: 70.17% - Val accuracy:  69.20%
Epoch: 003/010 - Loss:  0.557834
Train accuracy: 84.82% - Val accuracy:  82.34%
Epoch: 004/010 - Loss:  0.417997
Train accuracy: 86.81% - Val accuracy:  83.80%
Epoch: 005/010 - Loss:  0.365106
Train accuracy: 90.61% - Val accuracy:  86.34%
Epoch: 006/010 - Loss:  0.335922
Train accuracy: 92.51% - Val accuracy:  87.91%
Epoch: 007/010 - Loss:  0.310590
Train accuracy: 93.85% - Val accuracy:  88.23%
Epoch: 008/010 - Loss:  0.295640
Train accuracy: 95.19% - Val accuracy:  87.63%
Epoch: 009/010 - Loss:  0.273870
Train accuracy: 94.28% - Val accuracy:  86.66%
Epoch: 010/010 - Loss:  0.256638
Train accuracy: 95.05% - Val accuracy:  86.66%


### **Test**

In [69]:
print(f'Test accuracy: {accuracy(classifier, test_dataloader)*100:.2f}%')

Test accuracy: 86.93%
